# MVP — Mesa de Servicio Inteligente (IX Comercio + Intcomex)

Genera `resultados.json` con tres vistas filtrables (**Todo / IX Comercio / Intcomex**).
Empresa por la `Clave`: `ITSD`=Intcomex; el resto=IX Comercio. Riesgo = **incumplimiento al cliente** (señales operativas). Ejecuta *Entorno de ejecución → Ejecutar todo*.

## 1. Drive + localizar la base (ruta directa, robusta)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
RUTA_DB='/content/drive/MyDrive/Especializacion IA/incidentes_anonimizado.db'
if not os.path.exists(RUTA_DB):
    validos=[c for c in glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True) if os.path.isfile(c)]
    if validos: RUTA_DB=max(validos,key=os.path.getsize)
print('USANDO:',RUTA_DB,'|',round(os.path.getsize(RUTA_DB)/1e6,1),'MB')

## 2. Cargar `issues`, marcar empresa, sentimiento e incumplimiento

In [ ]:
import sqlite3, pandas as pd, numpy as np, re, json
from datetime import datetime
con=sqlite3.connect(RUTA_DB)
tablas=pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)['name'].tolist()
print('Tablas:',tablas)
def load(n):
    if n in tablas:
        t=pd.read_sql('SELECT * FROM "%s"' % n,con); t.columns=t.columns.str.lower().str.strip(); return t
    return None

df=load('issues'); df=df.drop_duplicates()
print('issues:',df.shape)
clave='clave' if 'clave' in df.columns else df.columns[0]
df['empresa']=np.where(df[clave].astype(str).str.upper().str.startswith('ITSD'),'Intcomex','IX Comercio')
print(df['empresa'].value_counts())

COL_TEXTO='resumen' if 'resumen' in df.columns else df.select_dtypes('object').columns[0]
df[COL_TEXTO]=df[COL_TEXTO].astype(str).str.strip()
POS=set('gracias ok correcto solucionado resuelto disponible exitoso'.split())
NEG=set('error falla fallo sin no cancelacion cancelación pendiente caido caída bloqueo rechazo duplicado incidencia problema demora retraso urgente'.split())
def sent(t):
    pal=re.findall(r'[a-záéíóúñ]+',str(t).lower()); p=sum(w in POS for w in pal); n=sum(w in NEG for w in pal)
    return round((p-n)/(p+n),3) if (p+n) else 0.0
df['sentimiento']=df[COL_TEXTO].apply(sent)
df['frustracion']=((1-df['sentimiento'])/2*100).round(1)

estado='estado' if 'estado' in df.columns else None
cerrados=['resuelto','cerrado','cancelado','finalizado','listo','done','closed']
abierto=(~df[estado].astype(str).str.lower().isin(cerrados)).astype(int) if estado else pd.Series(1,index=df.index)
cols_prio=[c for c in ['prioridad','impacto','urgencia'] if c in df.columns]
prio_alto=(df[cols_prio].astype(str).apply(lambda col: col.str.lower().str.contains('alt|critic',regex=True)).any(axis=1).astype(int)) if cols_prio else pd.Series(0,index=df.index)
fecha='creada' if 'creada' in df.columns else None
if fecha:
    cre=pd.to_datetime(df[fecha],errors='coerce',utc=True); df['antiguedad_dias']=(pd.Timestamp.now(tz='UTC')-cre).dt.days.clip(lower=0).fillna(0)
else:
    df['antiguedad_dias']=0
q95=df['antiguedad_dias'].quantile(0.95) or 1
ant_norm=(df['antiguedad_dias']/q95).clip(0,1)
df['incumplimiento']=(100*(0.35*abierto+0.25*prio_alto+0.20*ant_norm+0.20*(df['frustracion']/100))).round(1)
df['banda']=pd.cut(df['incumplimiento'],[-1,35,60,101],labels=['bajo','medio','alto'])
df[['empresa','incumplimiento','frustracion']].describe(include='all')

## 3. BTI y operacional (`ingresadas`), con empresa

In [ ]:
ing=load('ingresadas')
if ing is not None:
    tk=next((c for c in ['ticket','llave','clave'] if c in ing.columns),None)
    ing['empresa']=np.where(ing[tk].astype(str).str.upper().str.startswith('ITSD'),'Intcomex','IX Comercio') if tk else 'IX Comercio'
    print('ingresadas:',ing.shape)
ordn=load('ordenes_diarias')
def bloque_bti_op(sub):
    bti=[]; op={}
    if sub is not None and len(sub):
        code=next((c for c in ['bti_clasificacion','clasificacion','bti'] if c in sub.columns),None)
        desc=next((c for c in ['resumen_problema','descripcion','detalle'] if c in sub.columns),None)
        if code:
            vc=sub[code].value_counts(); dmap={}
            if desc:
                for _,r in sub.iterrows(): dmap.setdefault(str(r[code]),str(r[desc]))
            bti=[{'codigo':str(k),'descripcion':dmap.get(str(k),'')[:90],'casos':int(v)} for k,v in vc.items()]
        aco=next((c for c in ['accion_ops','accion','estado'] if c in sub.columns),None)
        op['ingresadas_total']=int(len(sub))
        if aco: op['por_accion']={str(k):int(v) for k,v in sub[aco].value_counts().items()}
    return bti, op

## 4. Construir vistas (Todo / IX / Intcomex) — listas completas

In [ ]:
etq=['muy negativo','negativo','neutro','positivo','muy positivo']; bins=[-1,-0.6,-0.2,0.2,0.6,1.0001]
def recom(i): return 'Incumplimiento inminente: contacto y mitigacion ya' if i>=60 else ('Vigilar y priorizar' if i>=35 else 'Monitoreo estandar')
lab='prioridad' if 'prioridad' in df.columns else (estado or COL_TEXTO)
def construir_vista(sub, sub_ing):
    if len(sub)==0: return {'kpis':{},'clasificacion':{},'distribucion_riesgo':{},'sentimiento_hist':{'bins':etq,'conteo':[0]*5},'por_estado':{},'serie_mensual':{},'incumplimiento':[],'bti_tabla':[],'operacional':{},'gerencia':{}}
    hist=pd.cut(sub['sentimiento'],bins=bins,labels=etq,include_lowest=True).value_counts().reindex(etq).fillna(0).astype(int)
    sm={}
    if fecha:
        s=pd.to_datetime(sub[fecha],errors='coerce',utc=True).dt.to_period('M').astype(str).value_counts(); s=s[s.index!='NaT'].sort_index(); sm={str(k):int(v) for k,v in s.items()}
    det=sub.sort_values('incumplimiento',ascending=False).head(300)
    incump=[{'id':str(r[clave])[:18],'empresa':r['empresa'],'prioridad':(str(r[lab])[:20] if lab in sub.columns else ''),'estado':(str(r[estado])[:18] if estado else ''),'dias':int(r['antiguedad_dias']),'frustracion':float(r['frustracion']),'incumplimiento':float(r['incumplimiento']),'recomendacion':recom(r['incumplimiento']),'resumen':str(r[COL_TEXTO])[:80]} for _,r in det.iterrows()]
    bti,op=bloque_bti_op(sub_ing)
    if ordn is not None and len(sub)==len(df): op['ordenes_total']=int(len(ordn))
    ger={}
    if 'país' in sub.columns: ger['por_pais']={str(k):int(v) for k,v in sub['país'].value_counts().items()}
    mcol=next((c for c in sub.columns if 'marca' in c),None)
    if mcol: ger['por_marca']={str(k):int(v) for k,v in sub[mcol].value_counts().head(30).items()}
    ger['por_empresa']={str(k):int(v) for k,v in sub['empresa'].value_counts().items()}
    ger['pct_incumplimiento_alto']=round(float((sub['banda']=='alto').mean()*100),1)
    return {
        'kpis':{'total_tickets':int(len(sub)),'abiertos':int((sub['banda']!='bajo').sum()),'pct_alto_riesgo':round(float((sub['banda']=='alto').mean()*100),1),'frustracion_promedio':round(float(sub['frustracion'].mean()),1),'sentimiento_promedio':round(float(sub['sentimiento'].mean()),3),'incumplimiento_promedio':round(float(sub['incumplimiento'].mean()),1)},
        'clasificacion':({str(k)[:30]:int(v) for k,v in sub[lab].value_counts().items()} if lab in sub.columns else {}),
        'distribucion_riesgo':{k:int(v) for k,v in sub['banda'].value_counts().reindex(['bajo','medio','alto']).fillna(0).astype(int).items()},
        'sentimiento_hist':{'bins':etq,'conteo':hist.tolist()},
        'por_estado':({str(k)[:25]:int(v) for k,v in sub[estado].value_counts().items()} if estado else {}),
        'serie_mensual':sm,'incumplimiento':incump,'bti_tabla':bti,'operacional':op,'gerencia':ger}
ing_de=lambda nom: (ing[ing['empresa']==nom] if (ing is not None and nom!='Todo') else ing)
vistas={'Todo':construir_vista(df,ing),'IX Comercio':construir_vista(df[df['empresa']=='IX Comercio'],ing_de('IX Comercio')),'Intcomex':construir_vista(df[df['empresa']=='Intcomex'],ing_de('Intcomex'))}
resultados={'generado':datetime.now().strftime('%Y-%m-%d %H:%M')+' (IX + Intcomex, anonimizado)','empresas':list(vistas.keys()),'vistas':vistas}
with open('resultados.json','w',encoding='utf-8') as f: json.dump(resultados,f,ensure_ascii=False,indent=2)
for n,v in vistas.items(): print(n,'->',v['kpis'])
print('resultados.json listo.')

## 5. Publicar al dashboard (GitHub API) — con Colab Secrets
Token en Colab Secrets (llave 🔑): Name `GH_TOKEN`, Value tu token, y activa *Notebook access*. Si no existe, lo pide con getpass.

In [ ]:
import base64, requests
GH_OWNER='ricardocasallas3-ux'; GH_REPO='mesa-servicio-ia'; GH_PATH='docs/data/resultados.json'; GH_BRANCH='main'
token=None
try:
    from google.colab import userdata; token=userdata.get('GH_TOKEN')
except Exception as e:
    print('Sin secret GH_TOKEN (', e, ')')
if not token:
    import getpass; token=getpass.getpass('Pega tu token de GitHub: ')
headers={'Authorization':'token '+token,'Accept':'application/vnd.github+json'}
url='https://api.github.com/repos/%s/%s/contents/%s'%(GH_OWNER,GH_REPO,GH_PATH)
r=requests.get(url,headers=headers,params={'ref':GH_BRANCH}); sha=r.json().get('sha') if r.status_code==200 else None
with open('resultados.json','rb') as f: contenido=base64.b64encode(f.read()).decode()
payload={'message':'Actualizar dashboard (IX+Intcomex) desde Colab','content':contenido,'branch':GH_BRANCH}
if sha: payload['sha']=sha
resp=requests.put(url,headers=headers,json=payload)
print('OK https://%s.github.io/%s/'%(GH_OWNER,GH_REPO) if resp.status_code in (200,201) else ('Error %s: %s'%(resp.status_code,resp.json())))